> VIGOR 2.12.1 example. Install with `pip install "VIGOR[io,progress,plot] @ git+https://github.com/sanchestm/VIGOR"` (see the [README](../../README.md)). Outputs are cleared: this notebook was checked against the 2.12.1 API but not re-executed for this release.

# Real26: covariance, factors, heldout prediction and bounded discovery

Individually migrated from `mega_gcorr_domino_2_11_4(6).ipynb`. See `MIGRATION.md` for original-cell lineage and deferred stages.

The default main path fits all **26 actual traits and 7,616 animals**, with deterministic whole-animal and within-animal holdouts. It uses the pinned **r11.2.1** transductive GRM; a new r11.3 cohort needs its own paired fixture and cache. Trait scaling and selection use observed training data only. Previously inspected split and upstream pre-regression limit scientific inference.

Set `VIGOR_RUN_EXTENDED=1` for bounded real26 factorGWAS, fine mapping and covariance exploration. This does not run a whole genome/server mega analysis. Optional stages are reported separately; disabled branches are skipped, never counted as scientific tests. No Optuna search runs by default.

In [ ]:
from pathlib import Path
import os, sys
# Optional server overrides: VIGOR_SOURCE, VIGOR_REAL26_FIXTURE, VIGOR_PLINK_PREFIX,
# VIGOR_GDEC, VIGOR_RSCRIPT, VIGOR_OUTPUT, VIGOR_THREADS (1..20).
SOURCE = Path(os.environ.get('VIGOR_SOURCE', Path.cwd().parents[1])).resolve()
if not (SOURCE/'vigor'/'__init__.py').is_file():
    candidates = [Path.cwd(), *Path.cwd().parents]
    SOURCE = next(p for p in candidates if (p/'vigor'/'__init__.py').is_file())
sys.path.insert(0, str(SOURCE))
for name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ[name] = os.environ.get('VIGOR_THREADS', '2')
import numpy as np, pandas as pd
get_ipython().run_line_magic('matplotlib', 'inline')
import matplotlib.pyplot as plt
import vigor
from examples.real26_workflow import (configuration, load_fixture, run_prediction,
    run_comparators, run_extended, write_json)
assert Path(vigor.__file__).resolve() == SOURCE/'vigor'/'__init__.py'
print({'version':vigor.__version__, 'source':'selected candidate', 'threads':os.environ['OMP_NUM_THREADS']})


In [ ]:
config = configuration('mega')
data = load_fixture(config)
print({'shape':data['ztrain'].shape, 'heldout_cells':{k:int(v.sum()) for k,v in data['masks'].items()}, 'assembly':'r11.2.1'})

## Current production covariance selection and prediction
Sparse G/E rank-grid CV uses only masked training traits. Inspect failed ranks; a successful full-rank fit is not proof that every candidate converged. Inference G/E and prediction profiles serve different purposes. The inherited residual-retention scalar has project-specific provenance; no claim of universal optimality is made.

In [ ]:
prediction = run_prediction(data, config)
fit = prediction['covariance_fit']
selection = prediction['prediction_sparse_cv']
display(pd.DataFrame(selection['candidates'])[['state','config']])
metrics = pd.read_csv(config['output']/'vigor_metrics.csv')
display(metrics.groupby(['target','split']).predictive_r2.mean().rename('macro_predictive_R2').to_frame())

## Saved native covariance and observed-trait embeddings
Save/reopen through VIGOR helpers, retaining the phenotype digest. Own-phenotype factor projections are **not** kinship-informed breeding values. Missing animals receive prior-mean scores. These diagnostic plots do not enter selection or holdout scoring.

In [ ]:
from vigor.phenome import (save_covariance_fit, open_covariance_fit, native_factor_loadings, animal_factor_embeddings)
checkpoint = config['output']/'covariance.zarr'
save_covariance_fit(fit, data['ztrain'], checkpoint, parameters={'selection':selection['chosen']})
reopened = open_covariance_fit(checkpoint)
np.testing.assert_allclose(reopened['G_prediction'], fit['G_prediction'])
G_loadings = native_factor_loadings(reopened,'G')
E_loadings = native_factor_loadings(reopened,'E')
embeddings = animal_factor_embeddings(data['ztrain'], reopened, center=False, blas_threads=config['threads'])
assert embeddings.sizes['sample'] == len(data['ztrain'])
print({'native_G_factors':G_loadings.shape[1], 'native_E_factors':E_loadings.shape[1], 'embeddings':'observed-trait projection only'})

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(12,4))
for axis,component in zip(ax,['G','E']):
    matrix = np.asarray(reopened[component])
    scales = np.sqrt(np.maximum(np.diag(matrix),1e-12))
    image = axis.imshow(matrix/scales[:,None]/scales[None,:],vmin=-1,vmax=1,cmap='RdBu_r')
    axis.set_title(component+' correlation, all 26 traits')
fig.colorbar(image,ax=ax.tolist(),shrink=.7)
fig.savefig(config['output']/'GE_correlations.png',dpi=140)
plt.show()

## Sign-invariant clustering of native loadings
Cluster normalized absolute G loadings using Ward linkage; this preserves trait-group exploration without inventing ontology labels. External annotation enrichment and its multiple-testing/permutation controls remain unrun until annotations are supplied.

In [ ]:
from scipy.cluster.hierarchy import linkage, leaves_list, fcluster
absolute = np.abs(G_loadings.to_numpy(float))
normalized = absolute/np.maximum(np.linalg.norm(absolute,axis=1,keepdims=True),1e-12)
linked = linkage(normalized,method='ward')
order = leaves_list(linked)
clusters = pd.DataFrame({'trait':G_loadings.index,'cluster':fcluster(linked,5,criterion='maxclust')})
clusters.to_csv(config['output']/'native_loading_clusters.csv',index=False)
fig,ax = plt.subplots(figsize=(8,5))
ax.imshow(normalized[order],aspect='auto',cmap='viridis')
ax.set_title('Normalized absolute native G loadings, Ward trait order')
ax.set_xlabel('Native factor'); ax.set_ylabel('Trait (cluster order)')
fig.savefig(config['output']/'native_G_loading_clusters.png',dpi=140)
plt.show()

In [ ]:
# The prediction entrypoint executes both genuine comparators.
# Enable here too when this independent workflow needs a fresh matched comparison.
comparators = run_comparators(data, config, prediction)
if len(comparators):
    display(comparators.groupby(['model','target','split']).predictive_r2.mean().to_frame())

## Explicit bounded optional scientific stages
With the toggle enabled, use all 26 traits on 512 selected animals and 128 real markers in two windows. Marker selection is phenotype-independent; factorGWAS refits LOCO backgrounds on this panel. Fine mapping uses 24 markers and observed training outcomes for one predeclared-by-training-coverage trait. This is an execution example, without genome-wide calibration, significance or causal-identification claims.

In [ ]:
extended = run_extended(data, config, prediction)
print(extended)

## Optional large analyses and scientific limitations
Full-server genome scans, graph/haplotype fine mapping, SHAP, annotation/ontology enrichment, additional A/D/GxE kernels, bootstrap coverage and Optuna are not enabled by this bounded run. Ontology evaluation requires supplied trait annotations; no labels are invented. Plotting/native-factor and holdout-comparison intent are retained above; historical detailed exploratory cell intent and locations are in MIGRATION.md. Fixed-support analytic SE excludes selection uncertainty; any future SE run must use the exact original phenotype digest and record probes. These stages remain **unrun**, not validated by executing their disabled branches.

In [ ]:
write_json(config['output']/'notebook_summary.json', {'state':'COMPLETE','shape':list(data['ztrain'].shape),'extended_state':extended['state'],'assembly':'r11.2.1','unrun':['whole-genome/server scale','ontology without annotations','SHAP','new A/D/GxE kernels','analytic SE selection uncertainty']})
print('Completed real26 workflow; this remains an unapproved candidate.')